# 04 — Feature Engineering
Task 5: add historical price behavior (lags, rolling stats, momentum, seasonality) to the clean dataset.

## 1. Load the clean dataset

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("../data/processed/tomato_clean.csv")

df["date"] = pd.to_datetime(df["date"])

print("Shape:", df.shape)
df.head()

## 2. Check your columns
If your actual column names differ from what's expected below, don't rename blindly — check the printed list first.

In [ ]:
print(df.columns.tolist())

## 3. Sort by market and date
Critical: keeps each market's price history as its own sequence instead of mixing markets together.

In [ ]:
df = df.sort_values(
    ["market", "date"]
).reset_index(drop=True)

## 4. Create lag features

**Previous day**

In [ ]:
df["price_lag_1"] = (
    df.groupby("market")["modal_price"]
    .shift(1)
)

**Previous 7 observations**

In [ ]:
df["price_lag_7"] = (
    df.groupby("market")["modal_price"]
    .shift(7)
)

**Previous 14 observations**

In [ ]:
df["price_lag_14"] = (
    df.groupby("market")["modal_price"]
    .shift(14)
)

**Previous 30 observations**

In [ ]:
df["price_lag_30"] = (
    df.groupby("market")["modal_price"]
    .shift(30)
)

## 5. Create rolling features
Lags give one previous value; rolling features capture the recent trend.

**7-period moving average**

In [ ]:
df["rolling_mean_7"] = (
    df.groupby("market")["modal_price"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

**14-period moving average**

In [ ]:
df["rolling_mean_14"] = (
    df.groupby("market")["modal_price"]
    .transform(
        lambda x: x.shift(1).rolling(14).mean()
    )
)

**30-period moving average**

`shift(1)` is essential here — without it, the rolling window would include today's own price and leak the target into the features.

In [ ]:
df["rolling_mean_30"] = (
    df.groupby("market")["modal_price"]
    .transform(
        lambda x: x.shift(1).rolling(30).mean()
    )
)

## 6. Create rolling volatility
Low volatility → relatively stable market. High volatility → unpredictable market. This will eventually help Kadak Bhav estimate price risk.

In [ ]:
df["rolling_std_7"] = (
    df.groupby("market")["modal_price"]
    .transform(
        lambda x: x.shift(1).rolling(7).std()
    )
)

## 7. Create price momentum
Whether prices are trending up or down. Positive → increasing, negative → decreasing.

In [ ]:
df["price_change_1"] = (
    df["price_lag_1"] - df["price_lag_7"]
)

In [ ]:
df["price_change_pct"] = (
    (
        df["price_lag_1"]
        - df["price_lag_7"]
    )
    / df["price_lag_7"]
) * 100

## 8. Add arrival information (if available)
High arrivals can signal supply pressure and possible price pressure — but this is a hypothesis for the model to learn, not a guaranteed relationship.

In [ ]:
if "arrival_quantity" in df.columns:
    df["arrival_lag_1"] = (
        df.groupby("market")["arrival_quantity"]
        .shift(1)
    )

    df["arrival_rolling_7"] = (
        df.groupby("market")["arrival_quantity"]
        .transform(
            lambda x: x.shift(1).rolling(7).mean()
        )
    )
else:
    print("No 'arrival_quantity' column found — skipping arrival features.")

## 9. Add seasonality
Cyclical month encoding so December and January are recognized as adjacent in the annual cycle.

In [ ]:
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

## 10. Check the new features

In [ ]:
new_features = [
    "price_lag_1",
    "price_lag_7",
    "price_lag_14",
    "price_lag_30",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_std_7",
    "price_change_1",
    "price_change_pct",
    "month_sin",
    "month_cos"
]

df[new_features].head(15)

## 11. Check missing values
Lag and rolling features naturally create missing values at the start of each market's history (e.g. `price_lag_30` can't exist for the first 30 rows of a market).

In [ ]:
df[new_features].isnull().sum()

## 12. Remove rows without enough history

In [ ]:
df = df.dropna(
    subset=[
        "price_lag_30",
        "rolling_mean_30",
        "rolling_std_7"
    ]
)

print("Final shape:", df.shape)

## 13. Define the final feature set

In [ ]:
FEATURES = [
    "min_price",
    "max_price",
    "average_price",
    "price_range",

    "year",
    "month",
    "day",
    "day_of_week",

    "month_sin",
    "month_cos",

    "price_lag_1",
    "price_lag_7",
    "price_lag_14",
    "price_lag_30",

    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",

    "rolling_std_7",

    "price_change_1",
    "price_change_pct"
]

if "arrival_quantity" in df.columns:
    FEATURES += [
        "arrival_quantity",
        "arrival_lag_1",
        "arrival_rolling_7"
    ]

print("Number of features:", len(FEATURES))
print(FEATURES)

## 14. Create the ML dataset

In [ ]:
TARGET = "modal_price"

model_df = df[
    FEATURES + [TARGET, "date", "market"]
].copy()

print(model_df.shape)
model_df.head()

## 15. Check for infinite values
`price_change_pct` can divide by a zero `price_lag_7`, so check for and clean up any resulting infinities.

In [ ]:
numeric_columns = model_df.select_dtypes(
    include=np.number
).columns

print(
    "Infinite values:",
    np.isinf(
        model_df[numeric_columns]
    ).sum().sum()
)

In [ ]:
model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

model_df = model_df.dropna(
    subset=FEATURES + [TARGET]
)

print("Shape after cleanup:", model_df.shape)

## 16. Save the feature-engineered dataset

In [ ]:
os.makedirs(
    "../data/processed",
    exist_ok=True
)

output_path = (
    "../data/processed/"
    "tomato_features.csv"
)

model_df.to_csv(
    output_path,
    index=False
)

print(
    f"Saved: {output_path}"
)

## 17. Visualize the new historical features
Sanity-check that the rolling average tracks the actual price sensibly for one market.

In [ ]:
import matplotlib.pyplot as plt

market_name = model_df["market"].iloc[0]

market_data = (
    model_df[
        model_df["market"] == market_name
    ]
    .sort_values("date")
)

plt.figure(figsize=(12, 5))

plt.plot(
    market_data["date"],
    market_data["modal_price"],
    label="Actual Price"
)

plt.plot(
    market_data["date"],
    market_data["rolling_mean_7"],
    label="7-Period Average"
)

plt.xlabel("Date")
plt.ylabel("Price")
plt.title(
    f"Price Trend - {market_name}"
)

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## ⚠️ Data leakage reminder
All rolling/lag features use `shift(1)` before the window so today's own price is never used to predict today. This is why we use `x.shift(1).rolling(...)` instead of `x.rolling(...)`.

## Next step
Once `tomato_features.csv` is saved and the graph looks sensible, Task 6 retrains XGBoost on this richer feature set and compares MAE/RMSE/MAPE/R² against both the Task 4 baseline and the simple-feature XGBoost run.